# landing_eda_pncp — EDA PNCP (Portal Nacional de Contratações Públicas)

> **Exploration only — not part of the production pipeline.**

## Purpose
Understand structure, quality, and anomalies in the PNCP raw JSON files
before defining and validating the Bronze schema.

## Central question
*"What is in the PNCP raw data, how are nested objects structured,
and what design decisions does that force in Bronze?"*

## Engine
DuckDB — same engine used in Bronze pipeline. No pandas.

## Steps
1. Imports and configuration
2. File inventory
3. Schema discovery
4. Key field analysis
5. Nested objects inspection
6. Inconsistencies and design decisions
7. Dynamic summary


## Step 1 — Imports and configuration

In [ ]:
import sys
from pathlib import Path

# --- Resolve root ------------------------------------------------------------
try:
    _nb_file = Path(__vsc_ipynb_file__).resolve()
    _root_candidate = _nb_file.parent.parent.parent  # eda/ → notebooks/ → fornecedor360-local/
except NameError:
    _root_candidate = Path.cwd()
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
        if (candidate / "utils" / "paths.py").exists():
            _root_candidate = candidate
            break

sys.path.insert(0, str(_root_candidate / "utils"))

from paths        import get_project_root, to_sql_path
from duckdb_utils import get_connection

PROJECT_ROOT = get_project_root()
con = get_connection()  # in-memory — EDA never writes

PNCP_ROOT      = PROJECT_ROOT / "data" / "raw" / "pncp"
available_files = sorted(PNCP_ROOT.glob("*.json")) if PNCP_ROOT.exists() else []
SAMPLE_FILE     = available_files[0]  if available_files else None
SAMPLE_PATH     = to_sql_path(SAMPLE_FILE) if SAMPLE_FILE else None

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"PNCP_ROOT     : {PNCP_ROOT}")
print(f"Files found   : {len(available_files)}")
if available_files:
    print(f"Sample file   : {SAMPLE_FILE.name}")

## Step 2 — File inventory

**Question:** *How many months do I have? What is the record count and size per file?
Are all files non-empty?*


In [ ]:
print("=== File Inventory ===")
print()
print(f"{'file':<20} {'size_mb':>8} {'records':>12}")
print("-" * 44)

total_mb, total_rec = 0.0, 0
for f in available_files:
    size_mb = f.stat().st_size / 1e6
    path    = to_sql_path(f)
    records = con.execute(f"SELECT COUNT(*) FROM read_json_auto('{path}')").fetchone()[0]
    total_mb  += size_mb
    total_rec += records
    print(f"{f.name:<20} {size_mb:>8.1f} {records:>12,}")

print("-" * 44)
print(f"{'TOTAL':<20} {total_mb:>8.1f} {total_rec:>12,}")


## Step 3 — Schema discovery

**Question:** *What columns exist in the PNCP JSON? Which are nested (STRUCT/LIST)?
What are the null rates? What needs special handling in Bronze?*


In [ ]:
print(f"=== Schema Discovery — {SAMPLE_FILE.name} ===")
print()

schema = con.execute(f"DESCRIBE SELECT * FROM read_json_auto('{SAMPLE_PATH}') LIMIT 0").fetchall()
total  = con.execute(f"SELECT COUNT(*) FROM read_json_auto('{SAMPLE_PATH}')").fetchone()[0]

print(f"Records : {total:,}  |  Columns : {len(schema)}")
print()
print(f"{'column':<50} {'type':<20} {'nulls':>8} {'null%':>6}")
print("-" * 90)

for col, dtype, *_ in schema:
    if "STRUCT" in dtype.upper() or "MAP" in dtype.upper() or "LIST" in dtype.upper():
        print(f"{col:<50} {dtype[:20]:<20} {'(nested)':>8}")
        continue
    nulls = con.execute(
        f"SELECT COUNT(*) FROM read_json_auto('{SAMPLE_PATH}') WHERE \"{col}\" IS NULL"
    ).fetchone()[0]
    pct = nulls / total * 100 if total else 0
    print(f"{col:<50} {dtype[:20]:<20} {nulls:>8,} {pct:>5.1f}%")


## Step 4 — Key field analysis

**Question:** *Is niFornecedor always a CNPJ? Are there CPF (PF), foreign suppliers (PE)?
What is the tipoPessoa distribution? Which records should Silver filter out?*


In [ ]:
print("=== Key Field Analysis ===")
print()

print("--- niFornecedor + tipoPessoa ---")
con.execute(f"""
    SELECT
        tipoPessoa,
        COUNT(*)                                                      AS total,
        COUNT(*) FILTER (WHERE niFornecedor IS NULL)                  AS ni_nulls,
        COUNT(*) FILTER (WHERE LENGTH(niFornecedor) = 14)             AS len_14_cnpj,
        COUNT(*) FILTER (WHERE LENGTH(niFornecedor) = 11)             AS len_11_cpf,
        COUNT(*) FILTER (WHERE LENGTH(niFornecedor) NOT IN (11,14))   AS other_len
    FROM read_json_auto('{SAMPLE_PATH}')
    GROUP BY tipoPessoa ORDER BY total DESC
""").df()

print()
print("--- valor fields ---")
con.execute(f"""
    SELECT
        COUNT(*) FILTER (WHERE valorInicial IS NULL)             AS vi_nulls,
        MIN(TRY_CAST(valorInicial AS DOUBLE))                   AS vi_min,
        MAX(TRY_CAST(valorInicial AS DOUBLE))                   AS vi_max,
        COUNT(*) FILTER (WHERE TRY_CAST(valorInicial AS DOUBLE) < 0) AS vi_negative
    FROM read_json_auto('{SAMPLE_PATH}')
""").df()


## Step 5 — Nested objects inspection

**Question:** *What nested objects (STRUCT/LIST) does DuckDB detect?
How many records have null nested objects?
What fields are inside each nested object that Bronze needs to flatten?*


In [ ]:
print("=== Nested Objects ===")
print()

schema_dict = {col: dtype for col, dtype, *_ in schema}
nested_cols = [(col, dtype) for col, dtype in schema_dict.items()
               if any(t in dtype.upper() for t in ["STRUCT", "MAP", "LIST"])]

print(f"Nested columns: {[c for c, _ in nested_cols]}")
print()

for col, dtype in nested_cols:
    null_count = con.execute(
        f"SELECT COUNT(*) FROM read_json_auto('{SAMPLE_PATH}') WHERE \"{col}\" IS NULL"
    ).fetchone()[0]
    print(f"--- {col} ---")
    print(f"  Type        : {dtype[:100]}")
    print(f"  Null records: {null_count:,} / {total:,} ({null_count/total*100:.1f}%)")
    try:
        sample = con.execute(
            f"SELECT \"{col}\" FROM read_json_auto('{SAMPLE_PATH}') "
            f"WHERE \"{col}\" IS NOT NULL LIMIT 1"
        ).fetchone()
        if sample:
            print(f"  Sample      : {str(sample[0])[:120]}")
    except Exception as e:
        print(f"  Sample error: {e}")
    print()


## Step 6 — Inconsistencies and design decisions

**Question:** *What anomalies require explicit handling in Bronze?
What are the key design decisions and their rationale?*


In [ ]:
print("=== Inconsistencies and Design Decisions ===")
print()

print("1. niFornecedor — mixed types (CNPJ=PJ, CPF=PF, other=PE)")
print("   14=CNPJ(PJ) | 11=CPF(PF — LGPD protected) | 2=PE(foreign, no CNPJ)")
print("   Bronze: STRING — alphanumeric CNPJ incoming July 2026 (ADR-002)")
print("   Silver: filter tipoPessoa = 'PJ' for supplier dimension")
print()

print("2. tipoPessoa — three values: PJ, PF, PE")
print("   PE = Pessoa Estrangeira — no CNPJ, no CPF")
print("   Bronze: keep ALL. Silver: filter.")
print()

print("3. codigoPaisFornecedor — ISO 3166-1 alpha-3 (BRA, USA, DEU)")
null_pais = con.execute(
    f"SELECT COUNT(*) FROM read_json_auto('{SAMPLE_PATH}') WHERE codigoPaisFornecedor IS NULL"
).fetchone()[0]
print(f"   Null rate: {null_pais:,} / {total:,} ({null_pais/total*100:.1f}%) — expected for PJ (Brazil-only)")
print()

print("4. Nested objects — Bronze strategy")
for col, dtype in nested_cols:
    print(f"   {col}: CAST each subfield to STRING individually")
print()

print("5. Duplicate check on numeroControlePNCP")
dup = con.execute(
    f"SELECT COUNT(*) - COUNT(DISTINCT numeroControlePNCP) AS duplicates "
    f"FROM read_json_auto('{SAMPLE_PATH}')"
).fetchone()[0]
print(f"   Duplicates: {dup:,}")
print()

print("6. esfera filter — Bronze keeps all, Silver validates esfera")
con.execute(
    f"SELECT orgaoEntidade.esferaId AS esfera, COUNT(*) AS cnt "
    f"FROM read_json_auto('{SAMPLE_PATH}') GROUP BY esfera ORDER BY cnt DESC LIMIT 5"
).df()


## Step 7 — Dynamic summary

**Question:** *What have I learned? What are the decisions Bronze depends on?*


In [ ]:
from IPython.display import display, Markdown

nested_names = [c for c, _ in nested_cols]
pj_count = con.execute(
    f"SELECT COUNT(*) FROM read_json_auto('{SAMPLE_PATH}') WHERE tipoPessoa = 'PJ'"
).fetchone()[0]
pj_pct = pj_count / total * 100 if total else 0

summary = f"""
## PNCP — EDA Summary

| | |
|---|---|
| Files available | {len(available_files)} ({available_files[0].stem if available_files else 'N/A'} → {available_files[-1].stem if available_files else 'N/A'}) |
| Sample records | {total:,} ({SAMPLE_FILE.name}) |
| Total columns | {len(schema)} |
| Nested columns | {len(nested_cols)}: {', '.join(nested_names)} |
| PJ records (Silver scope) | {pj_count:,} ({pj_pct:.1f}%) |

### Key Bronze decisions
| Decision | Reason |
|---|---|
| All scalar columns as STRING | No type guarantees across months |
| Nested structs: CAST each subfield to STRING | DuckDB infers STRUCT — cast individually |
| Keep PF and PE records | Bronze never filters — Silver applies tipoPessoa filter |
| Partition by `_year_month` | Monthly loading from 2021 onwards |
| `_esfera_valida` flag | Validates unidadeOrgao.esfera — national scope |
"""

display(Markdown(summary))
